# Agent란?
- LLM이 스스로 판단해서 어떤 행동(룰 사용 포함)을 할 지 결정하는 실행 주체를 의미합니다

In [1]:
from dotenv import load_dotenv
from langchain_openai.chat_models.base import ChatOpenAI
import os

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-2t69LdIulCeS


In [6]:
# 5,243.38

prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

In [3]:
chat = ChatOpenAI(temperature=0.1)

In [4]:
result = chat.invoke(prompt)

In [5]:
result.content

'The total cost is $4,363.38.'

In [6]:
chat = ChatOpenAI(model="gpt-4o", temperature=0.1)

In [7]:
result = chat.invoke(prompt)

In [9]:
print(result.content)

To find the total cost, you need to add all the amounts together:

\[ 
355.39 + 924.87 + 721.2 + 1940.29 + 573.63 + 65.72 + 35.00 + 522.00 + 76.16 + 29.12 = 5243.38 
\]

So, the total cost is $5243.38.


In [ ]:
"""
    프롬프트의 정답
    $4,363.38.

    계산기에서 직접 계산하기 ↓↓↓
    $5,243.38

    llm의 계산 착오
    ※이유※
    LLM은 산술 연산을 수행하지 않습니다. 이런 계산은 AI보다 계산기가 더 잘합니다.
    LLM은 text를 생성해내는 모델입니다. 문장의 시퀀스의 다음 token이 무엇인지 통계적으로 추측합니다.
    이러한 LLm의 오류를 잡기 위해서는 agent를 제공해주어야 합니다.
    그리고 agent를 위한 tool(툴)을 만들고, agent가 tool을 선택해서 실행하는 것입니다.
"""

# Agent 생성

In [2]:
# create_agent: 함수로 통일

from langchain.agents import create_agent
from langchain.tools import tool

In [20]:
@tool
def plus(num1: float, num2: float) -> float:
    """
        Adds two numbers and return the result.
    """
    return num1 + num2


"""
    json {
        "name": plus,
        "description": "Adds two numbers and return the result.",
        "parameters": {
            "num1": "float",
            "num2": "float",
        }
    
    }

"""

None

In [33]:
agent = create_agent(
    model = "gpt-3.5-turbo",
    tools=[plus],
    system_prompt="You are a helpful assistant"
)

In [34]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
             "content" : prompt
        }
    ]
})


In [35]:
result["messages"][-1].content

'The total cost is:\n- $355.39 + $924.87 = $1280.26\n- $721.20 + $1940.29 = $2661.49\n- $573.63 + $65.72 = $639.35\n- $35.00 + $522.00 = $557.00\n- $76.16 + $29.12 = $105.28\n\nAdding all these amounts together:\n$1280.26 + $2661.49 + $639.35 + $557.00 + $105.28 = $5243.38\n\nTherefore, the total cost is $5243.38.'

In [24]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'plus', 'args': {'num1': 355.39, 'num2': 924.87}, 'id': 'call_jfKTHTyi6QG6M7dlptCq0fQH', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 721.2, 'num2': 1940.29}, 'id': 'call_MfVWCMQHefRHR9OkQkPrSM4K', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 573.63, 'num2': 65.72}, 'id': 'call_Ag9bCAZs7FsvgU8GnGBvUmvH', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 35.0, 'num2': 522.0}, 'id': 'call_qyHul5pnVq7pFoDcjg5RoNja', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 76.16, 'num2': 29.12}, 'id': 'call_aG3MxzG9H4BgStRn8AjWC3D7', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 1280.26, 'num2': 2661.49}, 'id': 'call_u1wzrETL0jfGZB7EnfVau2bI', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 3941.75, 'num2': 639.35}, 'id': 'call_Pi7f8GUwYSAtx4N9t3hzc5jh', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 4581.1, 'num2': 557.0}, 'id': 'call_lUxX4YSoGccTYsirfoFUszws', 'type': 'tool_call'}
{'name': 'plus', 'args': {'num1': 5138.1, 'num2': 10

In [3]:
@tool
def total_sum(numbers: list[float]) -> float:
    """
        Adds a list of numbers and returns the total sum.
        Use this tool when you need to calculate the total of multiple numbers.
        Input should be a string representation of a list.
        Example: "[1, 2, 3]"
    """
    return sum(numbers)

In [4]:
agent = create_agent(
    model = "gpt-3.5-turbo",
    tools=[total_sum],
    system_prompt="You are a helpful assistant"
)

In [7]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

result = agent.invoke({
    "messages": [
        {
            "role": "user",
             "content" : prompt
        }
    ]
})


In [30]:
result["messages"][-1].content

'The total cost of $355.39 + $924.87 + $721.20 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12 is $5243.38.'

In [31]:
for message in result["messages"]:
    if message.__class__.__name__ == "AIMessage" and message.tool_calls:
        for i in message.tool_calls:
            print(i)

{'name': 'total_sum', 'args': {'numbers': [355.39, 924.87, 721.2, 1940.29, 573.63, 65.72, 35.0, 522.0, 76.16, 29.12]}, 'id': 'call_dluqmw76Ut6SUHmgfqMpxays', 'type': 'tool_call'}


# 랭스미스

# LangSmith(랭스미스)
- LLM 기반 애플리케이션의 디버깅, 성능 평가, 모니터링 등을 제공하는 랭체인의 통합 플랫폼입니다.

## 랭스미스 Open API Key 발급
- https://smith.langchain.com/ 접속
- 로그인 후 좌측 하단 [Setting] 메뉴 클릭
- [API Keys] 클릭 후 생성
- Description은 lang_ksh(이니셜)
- [default workspace]는 기존에 있는 workspace1로 만들고 생성
- 발급받은 KEY를 .env에 추가하기

### .env에 추가하기
- LANGCHAIN_TRACING_V2=true
- LANGCHAIN_ENDPOINT="https://api.smith.langchain.com"
- LANGCHAIN_PROJECT=lang_1900
- LANGSMITH_API_KEY=발급받은 랭스미스 key

### 설정 후 Jupyter Notebook 재실행

## Agent가 동작하는 과정
1. 끝날때까지 반복이 되는 loop입니다.
2. llm으로 부터 어떤 것을 할지(get action)을 받아온다. (lang smith의 output에서 확인가능)
3. 실행한 결과를 observation이라고 부른다. 다시 다음 next action을 실행시킨다.
4. Agent Finish를 응답받으면 마지막 action 값을 리턴한다.

## 1. ReAct(Reasoning and Action) Agent

In [3]:
from dotenv import load_dotenv
import os

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain_classic.agents import AgentExecutor, create_react_agent, create_openai_functions_agent
from langchain_classic import hub
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List #Python의 내장 모듈 typing

In [9]:
load_dotenv()

True

In [10]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0)

In [12]:
@tool
def plus(expression: str) -> float:
    """
        Adds multiple numbers and returns their total sum.

        The input must be a comma-spreated string of numbers.
        Example: "10,20,30"

        Use this tool when you nee to calcuate the sum of multiple values.
    """
    try:
        numbers = [float(num) for num in expression.split(",")]
        return sum(numbers)
    except Exception as e:
        return -1

In [18]:
tools = [plus]
react_agent_prompt = hub.pull("hwchase17/react")

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt = react_agent_prompt
)

#실행기
react_agent_executor = AgentExecutor(
    agent = agent,
    tools = tools,
    verbose = True #내부 동작 확인
)


In [19]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $1940.29 + $573.63 + $65.72 + $35.00 + $522.00 + $76.16 + $29.12"

result = react_agent_executor.invoke({
    "input":prompt
})



> Entering new AgentExecutor chain...
To find the total cost, I need to sum all the given amounts. I will use the plus tool to calculate the total.

Action: plus
Action Input: "355.39,924.87,721.2,1940.29,573.63,65.72,35.00,522.00,76.16,29.12"5243.38I now know the final answer
Final Answer: 5243.38

> Finished chain.


## 2. OpenAI Function Calling Agent

In [23]:
class CalculatorToolArgsSchema(BaseModel):
    numbers: List[float] = Field(description="Numbers to sum")

class CalculatorTool(BaseTool):
    # 약속된 필드 이름
    name: Type[str] = "calculator_tool"
    description: Type[str] = """
        Adds multiple numbers and returns their total sum.
        Use this tool when you need to calulate the sum of multiple values.
    """
    args_schema: Type[BaseModel] = CalculatorToolArgsSchema
    
    # BaseTool은 반드시 _run 함수를 재정의
    # tool을 호출했을 때 실행되는 메인로직
    def _run(self, numbers):
        return sum(numbers)

In [25]:
tools = [CalculatorTool()]

# placeholder(agent_scratchpad): 내부 tool, reasoning 호출 기록을 임시 저장
function_agent_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("placeholder", """{agent_scratchpad}"""),
])

#판단
agent = create_openai_functions_agent(
    llm=llm,
    tools=tools,
    prompt=function_agent_prompt
)

#실행기
calling_agent_executor = AgentExecutor(
    agent = agent,
    tools=tools,
    verbose=True
)

In [27]:
prompt = "cost of $355.39 + $924.87 + $721.2 + $19401.29 + $573.63 + $365.72 + $35.00 + $5522.00 + $76.216 + $29.12"

result = calling_agent_executor.invoke({
    "input":prompt
})



> Entering new AgentExecutor chain...

Invoking: `calculator_tool` with `{'numbers': [355.39, 924.87, 721.2, 19401.29, 573.63, 365.72, 35, 5522, 76.216, 29.12]}`


28004.436The total cost is $28,004.44.

> Finished chain.


In [2]:
result["output"]

NameError: name 'result' is not defined

# v1.0↑ create_agent(커스텀 툴)

In [4]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List , Tuple, Dict#Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-2t69LdIulCeS


In [8]:
tools = []

agent = create_agent (
    model="gpt-4o-mini",
    tools=[],
)

In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent

result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='c0989c88-7f15-46eb-88a7-73b708b6052d'),
  AIMessage(content='죄송하지만, 실시간 날씨 정보를 제공할 수는 없습니다. 하지만 강남 지역의 날씨를 확인하시려면 기상청 웹사이트나 날씨 앱을 이용하시면 됩니다. 현재 날씨에 대한 정보를 확인해 보세요!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 18, 'total_tokens': 71, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa46e05e5b', 'id': 'chatcmpl-Dh6zLoH6mgSADtlUTQy9qLBaTsak0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e3ea3-1a01-7911-a702-6d1ac119b4af-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 53, 'total_toke

In [22]:
result["messages"][-1].content

'죄송하지만, 실시간 날씨 정보를 제공할 수는 없습니다. 하지만 강남 지역의 날씨를 확인하시려면 기상청 웹사이트나 날씨 앱을 이용하시면 됩니다. 현재 날씨에 대한 정보를 확인해 보세요!'

In [41]:
# 지역 -> 위도, 경도
def get_coordinates(location_name):
    locator = Nominatim(user_agent="ksh")
    location = locator.geocode(location_name)

    return location.latitude, location.longitude, 

In [46]:
get_coordinates("선릉역")

(37.5057908, 127.0483487)

In [24]:
# 위도, 경도 -> 날씨

def get_weather(lat, lon):
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
    
    # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
    weather_codes = {
        0: "맑음 ☀️",
        1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
        45: "안개 🌫️", 48: "침강 안개 🌫️",
        51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
        61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
        71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
        80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
        95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
    }

    try:
        response = requests.get(url)
        datas = response.json()

        if "current_weather" in datas:
            current = datas["current_weather"]
            temp = current["temperature"]
            wind = current["windspeed"]
            code = current["weathercode"]

            condition = weather_codes.get(code, "알 수 없음")

            return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}"
            
        
    except Exception as e:
        return "요청 실패"

In [21]:
print(get_weather(37.500078, 127.035548))

상태: 흐림 ☁️
온도: 23.1°C
풍속: 5.4


# 함수 -> 툴로 변경 후 제공

In [53]:
class CoordinatesToolArgSchema(BaseModel):
    location_name: str = Field("위도와 경도로 바꾸고 싶은 장소명입니다.")

class CoordinatesTool(BaseTool):
    name: Type[str] = "coordinates_tool"
    description: Type[str] = """
        장소명을 위도(latitude)와 경도(longitude) 좌표로 변환합니다.
        장소명을 위도와 경도로 변환하고 싶을 대 사용하는 도구입니다.
    """
    args_schema: Type[BaseModel] = CoordinatesToolArgSchema

    def _run(self, location_name: str) -> Tuple[float, float]:
        locator = Nominatim(user_agent="ksh")
        location = locator.geocode(location_name)
    
        return location.latitude, location.longitude, 

In [54]:
class WeatherSearchToolArgSchema(BaseModel):
    lat: float = Field(description="위도, Example Value: 37.500078")
    lon: float = Field(description="경도, Example Value: 127.035548")
    
class WeatherSearchTool(BaseTool):
    name: Type[str] = "weather_search_tool"
    description: Type[str] = """
        지역의 날씨를 가져오고 싶을 때 사용하는 툴입니다.
        위도와 경도를 입력하면, 해당 지역의 날씨의 정보를 문자열로 반환합니다.
    """

    args_schema: Type[BaseModel] = WeatherSearchToolArgSchema
    
    # 위도, 경도 -> 날씨
    def _run(self, lat: float, lon: float) -> str:
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        
        # 기상 코드(WMO Code)를 한글로 변환하는 딕셔너리
        weather_codes = {
            0: "맑음 ☀️",
            1: "대체로 맑음 🌤️", 2: "구름 조금 ⛅", 3: "흐림 ☁️",
            45: "안개 🌫️", 48: "침강 안개 🌫️",
            51: "가벼운 이슬비 🌦️", 53: "이슬비 🌧️", 55: "강한 이슬비 ⛈️",
            61: "약한 비 💧", 63: "보통 비 ☔", 65: "강한 비 🌊",
            71: "약한 눈 ❄️", 73: "보통 눈 ☃️", 75: "강한 눈 🏔️",
            80: "약한 소나기 🌦️", 81: "보통 소나기 🌧️", 82: "강한 소나기 ⛈️",
            95: "뇌우 ⚡", 96: "뇌우 및 우박 ⛈️", 99: "심한 뇌우 🌪️"
        }
    
        try:
            response = requests.get(url)
            datas = response.json()
    
            if "current_weather" in datas:
                current = datas["current_weather"]
                temp = current["temperature"]
                wind = current["windspeed"]
                code = current["weathercode"]
    
                condition = weather_codes.get(code, "알 수 없음")
    
                return f"상태: {condition}\n온도: {temp}°C\n풍속: {wind}km/h"
            
        except Exception as e:
            return "요청 실패"

In [58]:
tools = [CoordinatesTool(),WeatherSearchTool()]

agent = create_agent (
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "강남의 현재 실시간 날씨 알려줘!")
])

chain = prompt | agent

result = chain.invoke({})

result

{'messages': [HumanMessage(content='강남의 현재 실시간 날씨 알려줘!', additional_kwargs={}, response_metadata={}, id='7ba5112d-57c2-4456-85ff-be6091122fa7'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 185, 'total_tokens': 201, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_aa46e05e5b', 'id': 'chatcmpl-Dh7vogIB7R71bhKWFeLeRwUcWkLbX', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e3eda-6b7b-7f92-b540-3731a6a94914-0', tool_calls=[{'name': 'coordinates_tool', 'args': {'location_name': '강남'}, 'id': 'call_HXVXhRU0IVHVeQz5NVFrT3j9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'out

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("human", "")
])

chain = prompt | agent

result = chain.invoke({})

result

1. DuckDuckgo search tool
   - 회사 정보를 웹에서 찾는 툴을 만들기
   - 회사가 상장했는가, 회사 주식 심볼(ticker, 티커)는 무엇인지?
   - 가령 A라는 회사에 대한 정보를 찾고자 한다면, agent에 의해 툴은 A가 어떤 회사인지 검색을 수행하게 한다

2. AlphaVantaga API(주식 회사 정보)
   - 1) 회사의 심볼을 알아내는 툴
   - 2) 손익 계산서를 위한 툴
   - 3) 뉴스 심리지수를 위한 툴
   - 4) 회사의 개요를 위한 툴

   - https://www.alphavantage.co/
   - 위 사이트에 접속 후 API_KEY 발급
   - 환경변수에 등록하기
   - ALPHA_VANTAGE_API_KEY="발급받은 키"

In [5]:
from dotenv import load_dotenv
import os
import requests

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.tools import tool, BaseTool
from langchain.agents import create_agent
from langchain_openai.chat_models.base import ChatOpenAI

from pydantic import BaseModel, Field
from typing import Any, Type, List, Tuple, Dict #Python의 내장 모듈 typing

# 추가된 import
from langchain_community.utilities.duckduckgo_search import DuckDuckGoSearchAPIWrapper
from geopy.geocoders import Nominatim

load_dotenv()
print(os.environ.get('OPENAI_API_KEY')[:20])

sk-proj-2t69LdIulCeS


## 1. 일, 주, 월 단위 실적을 제공
- https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol=IBM&apikey=demo

In [ ]:
"""
    {
        "Meta Data": {
        "1. Information": "Daily Prices (open, high, low, close) and Volumes",
        "2. Symbol": "IBM",
        "3. Last Refreshed": "2026-05-18",
        "4. Output Size": "Compact",
        "5. Time Zone": "US/Eastern"
        },
        "Time Series (Daily)": {
        "2026-05-18": {
        "1. open": "218.5500", # 시작가
        "2. high": "223.3300", # 최고가
        "3. low": "217.7500",  # 최저가
        "4. close": "222.7500", # 마감가
        "5. volume": "5946367"  # 거래량
        },
        "2026-05-15": {
        "1. open": "218.2000",
        "2. high": "220.9100",
        "3. low": "217.6150",
        "4. close": "219.3000",
        "5. volume": "6154450"
        },

"""

## 2. News & Sentiments
- https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers=AAPL&apikey=demo

In [ ]:
"""
    {
    "items": "50",

    # 뉴스의 지표
    "sentiment_score_definition": "x <= -0.35: Bearish; -0.35 < x <= -0.15: Somewhat-Bearish; -0.15 < x < 0.15: Neutral; 0.15 <= x < 0.35: Somewhat_Bullish; x >= 0.35: Bullish",
    "relevance_score_definition": "0 < x <= 1, with a higher score indicating higher relevance.",
    "feed": [
        {
        "title": "Apple CEO Tim Cook in Beijing with US Presidential Delegation – May 2026 - News and Statistics",
        "url": "https://www.indexbox.io/blog/tim-cook-joins-us-delegation-to-beijing-as-apple-navigates-china-ties/",
        "time_published": "20260519T031958",
        "authors": [],
        "summary": "Apple CEO Tim Cook is in Beijing as part of a U.S. presidential delegation, marking the first such visit in nearly a decade. This trip is crucial for Apple due to its extensive manufacturing operations in China and China being its largest market outside the U.S. Improved U.S.-China relations, including reduced tariffs and increased market access, would significantly benefit Apple, especially following Chinese President Xi Jinping's recent pledge to \"open wider\" for American businesses.",
        "banner_image": "https://www.indexbox.io/landing/img/blog/telegram-fallback/5eab958188a1a3d707e92b07af013cb6.webp",
        "source": "IndexBox",
        "category_within_source": "General",
        "source_domain": "IndexBox",
        "topics": [
        {
        "topic": "technology",
        "relevance_score": "0.812212"
        },
        {
        "topic": "economy_macro",
        "relevance_score": "0.745762"
        },
        {
        "topic": "finance",
        "relevance_score": "0.634877"
        },
        {
        "topic": "manufacturing",
        "relevance_score": "0.647957"
        }
        ],
        
        # 0.335609 : 뉴스 지표 점수 (Somewhat_Bullish)
        
        "overall_sentiment_score": 0.335609,
        "overall_sentiment_label": "Somewhat-Bullish",
        "ticker_sentiment": [
            {
            "ticker": "AAPL",
            "relevance_score": "1.000000",
            "ticker_sentiment_score": "0.423983",
            "ticker_sentiment_label": "Bullish"
            },
            {
            "ticker": "NVDA",
            "relevance_score": "0.610471",
            "ticker_sentiment_score": "0.315554",
            "ticker_sentiment_label": "Somewhat-Bullish"
            },
            {
            "ticker": "TSLA",
            "relevance_score": "0.611822",
            "ticker_sentiment_score": "0.327856",
            "ticker_sentiment_label": "Somewhat-Bullish"
            }
        ]
"""

## 3. 회사 재무 재표 손익(Income Statement)
- https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol=IBM&apikey=demo

In [ ]:
"""
{
    "symbol": "IBM",
    "annualReports": [
    {
        "fiscalDateEnding": "2025-12-31",
        "reportedCurrency": "USD",
        "grossProfit": "40185000000",    # 순순익
        "totalRevenue": "67535000000",   # 총매출
        "costOfRevenue": "27350000000",  # 원가
        "costofGoodsAndServicesSold": "27350000000",
        "operatingIncome": "10325000000",
        "sellingGeneralAndAdministrative": "18285000000",
        "researchAndDevelopment": "8320000000",
        "operatingExpenses": "29860000000",
        "investmentIncomeNet": "None",
        "netInterestIncome": "-1290000000",
        "interestIncome": "645000000",
        "interestExpense": "1935000000",
        "nonInterestIncome": "None",
        "otherNonOperatingIncome": "None",
        "depreciation": "None",
        "depreciationAndAmortization": "5021000000",
        "incomeBeforeTax": "10328000000",
        "incomeTaxExpense": "-242000000",
        "interestAndDebtExpense": "None",
        "netIncomeFromContinuingOperations": "10571000000",
        "comprehensiveIncomeNetOfTax": "None",
        "ebit": "12263000000",
        "ebitda": "17284000000",
        "netIncome": "10593000000"
        },

"""

### 4. 회사의 개요(Company Overview)
- https://www.alphavantage.co/query?function=OVERVIEW&symbol=IBM&apikey=demo

## Stock Tools 생성

In [12]:
class StockMartSymbolSearchToolArgSchema(BaseModel):
    query: str = Field(description="""
        The query you will search for Example query: Stock Market Symbol for Apple company.
    """)

class StockMartSymbolSearchTool(BaseTool):
    name: Type[str] = "stock_mark_symbol_search_tool"
    description: Type[str] = """
        Use this tool fin the stock market symbol for a company.
        It takes a query as an argument.
    """

    args_schema: Type[BaseModel] = StockMartSymbolSearchToolArgSchema
    
    def _run(self, query):
        ddg = DuckDuckGoSearchAPIWrapper()
        ddg.run(query)

In [20]:
def parse_output(result):
    return result["messages"][-1].content

In [21]:
tools = []

agent = create_agent (
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "엔비디아의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 엔비디아를 구매해야하는지 알려줘"
})

In [22]:
print(result)

엔비디아(NVIDIA Corporation)는 미국의 그래픽 처리 장치(GPU) 및 인공지능(AI) 칩 제조업체로, 특히 게임, 데이터 센터, 자율주행차, AI 및 머신러닝 분야에서 강력한 입지를 가지고 있습니다. 엔비디아의 주식 심볼은 **NVDA**입니다.

### 회사 개요
- **설립 연도**: 1993년
- **본사**: 미국 캘리포니아주 산타 클라라
- **주요 제품**: GeForce 그래픽 카드, Tesla AI 칩, NVIDIA DGX 시스템 등
- **시장 분야**: 게이밍, AI, 데이터 센터 및 클라우드 컴퓨팅, 자율주행차, 프로페셔널 비주얼라이제이션

### 손익 계산서
엔비디아의 손익 계산서는 공식 웹사이트나 금융 정보를 제공하는 플랫폼을 통해 확인할 수 있으나, 예를 들어 보통 매출 성장률, 순이익, 총 자산 수익률(ROA) 등이 주요 지표로 꼽힙니다. 이러한 지표들이 긍정적이라면 기업이 건전한 재무 상태임을 나타낼 수 있습니다.

### Recent News (뉴스)
엡솔루트 최신 뉴스는 다음과 같이 요약할 수 있습니다.
- **AI와 GPU 수요 증가**: 최근 몇 년 동안 AI 관련 수요가 급증함에 따라 엔비디아의 GPU도 각광받고 있습니다.
- **경쟁업체 분석**: AMD, Intel 등 다른 기업들과의 경쟁 상황은 엔비디아의 주가와 성과에 영향을 미칠 수 있습니다.
- **주가 변동성**: 기술주 특성상 주가가 변동성이 클 수 있으며, 시장 상황에 따라 크게 변동할 수 있습니다.

### 투자 결정
엔비디아에 투자할지 여부는 다음과 같은 요소를 고려해야 합니다:
1. **경쟁력**: 엔비디아는 AI와 게이밍 분야에서 강력한 경쟁력을 가지고 있으며, 이는 긍정적인 투자 지표입니다.
2. **재무 건강**: 손익 계산서를 통한 회사의 수익성 분석 및 부채 비율 등의 재무 지표를 고려해야 합니다.
3. **시장 동향**: AI와 관련된 트렌드가 지속적으로 확대되고 있는지, 이를 통해 성장 잠재력이 있는지 분석해야 합니다.
4. 

## News Sentiments Tool

In [6]:
alpha_vantage_api_key = os.environ.get("ALPHA_VANTAGE_API_KEY")
alpha_vantage_api_key

'6TI6KSVLLRPI4XYQ'

In [8]:
class NewsSentimentToolArgsSchema(BaseModel):
    symbol: str = Field(description = "Stock symbol of the company. Example: AAPL, TSLA")

class NewsSentimentTool(BaseTool):
    name: Type[str] = "news_sentiment_tool"
    description: Type[str] = """
        Use this to get the news sentiment of a company.
        You should enter a stock symbol.
    """

    args_schema: Type[BaseModel] = NewsSentimentToolArgsSchema

    def _run(self, symbol:str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas


In [27]:
tools = [
    StockMarkSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool() # 회사 최근 뉴스, 민감도 툴
]

agent = create_agent (
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "테슬라의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 테슬라를 구매해야하는지 알려줘"
})

In [28]:
print(result)

현재 테슬라(TSLA)의 주식 관련 정보를 살펴보면 다음과 같습니다:

### 1. **주식 심볼**
- 테슬라의 주식 심볼은 **TSLA**입니다.

### 2. **회사 개요**
테슬라는 전기차 및 청정 에너지 솔루션을 제조하고 판매하는 회사로, 혁신적 기술과 지속 가능한 에너지를 결합하여 자동차 산업의 변화를 선도하고 있습니다. 테슬라는 전기차 외에도 태양광 패널 및 에너지 저장 솔루션을 제공합니다. 이 회사는 자율주행 기술 개발에도 적극적으로 참여하고 있습니다.

### 3. **현재 뉴스 및 감정 분석**
현재 테슬라에 대한 뉴스 감정을 분석한 결과, 다음과 같은 기사들이 있습니다:

- **Bullish Sentiment** (긍정적 감정):
  - 여러 전문가들이 테슬라의 성장을 긍정적으로 평가하고 있으며, 미래의 성장 가능성을 보고 있습니다.
  - Jim Cramer는 테슬라가 배터리 기술의 우수성을 지니고 있다고 강조하며 Ford보다 뛰어나다고 언급했습니다.

- **Somewhat-Bullish Sentiment** (다소 긍정적 감정):
  - Barclays가 테슬라에 연계된 AutoCallable 노트를 제공하였고, 이로 인해 투자자들의 관심이 높아지고 있습니다.
  - Gene Munster는 테슬라와 SpaceX의 합병 가능성을 50% 이상으로 보고 있습니다.

- **Bearish Sentiment** (부정적 감정):
  - 테슬라의 로보택시와 관련된 사고가 보도되었고, 이에 대한 비판이 제기되고 있습니다.
  - 내부자 거래가 보고되었으며, 최근 주식 매도가 있었던 것으로 나타났습니다.

### 4. **결정 여부**
주식 구매 여부를 결정하기 위해 고려할 몇 가지 포인트는 다음과 같습니다:

- **경쟁 및 시장 전망**: 전기차 시장의 경쟁이 치열해지고 있지만, 테슬라는 여전히 기술적 우위를 가지고 있습니다. 그러나 최근의 부정적 소식이 투자자들에게 미치는 영향은 신중하게 감시해야 합니다.

- **성장 가능성**: 많은 

## 회사의 손익 계산서 툴(income statement)

In [13]:
class CompanyIncomeStatementToolArgsSchema(BaseModel):
    symbol: str = Field(description = "Stock symbol of the company. Example: AAPL, TSLA")

class CompanyIncomeStatementTool(BaseTool):
    name: Type[str] = "company_income_statement_tool"
    description: Type[str] = """
        Use this to get the income statement of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = NewsSentimentToolArgsSchema

    def _run(self, symbol:str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=INCOME_STATEMENT&symbol={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas


In [32]:
tools = [
    StockMarkSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool(), #회사의 재무재표 손익 툴
    
]

agent = create_agent (
    model="gpt-4o-mini",
    tools=tools,
)

prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}")
])

chain = prompt | agent | RunnableLambda(parse_output)

result = chain.invoke({
    "question": "테슬라의 심볼과 회사의 개요, 회사의 손익 계산서, 뉴스등을 고려해서 테슬라를 구매해야하는지 알려줘"
})

In [33]:
print(result)

### 테슬라(Tesla)의 투자 분석

#### 1. 테슬라 심볼
- **주식 심볼**: TSLA

#### 2. 회사 개요
테슬라(Tesla)는 전기차 제조 및 에너지 저장 솔루션의 선두주자입니다. 이 회사는 전 세계적으로 전기차, 배터리 저장 장치, 그리고 태양 에너지 제품을 제공하고 있으며, 혁신적인 기술과 환경 지속 가능성에 중점을 두고 있습니다.

#### 3. 손익 계산서 (Recent Reports)
최근 연간 재무 보고서에 따르면, 테슬라는 다음과 같은 성과를 보였습니다:

- **2023년 12월 31일 기준**
  - 총 수익: $96,773,000,000
  - 총 이익: $17,660,000,000
  - 운영 소득: $8,891,000,000
  - 순이익: $14,974,000,000

- **2024년 12월 31일 예측**
  - 예상 수익: $97,690,000,000
  - 예상 총 이익: $17,450,000,000
  - 예상 운영 소득: $7,076,000,000
  - 예상 순이익: $7,130,000,000

이러한 수치는 테슬라가 안정적인 수익과 이익률을 유지하고 있으며, 앞으로의 성장 가능성을 암시합니다.

#### 4. 뉴스 및 시장 감정
최근 투자자들 사이에서의 뉴스 감정은 비교적 **중립적**인 상황입니다:

- **일부 긍정적인 기사**:
  - 테슬라의 배터리 기술과 생산능력의 장점.
  - 일론 머스크는 테슬라와 스페이스X의 결합 가능성에 대해 언급하며 주가 상승의 여지를 강조.
  
- **부정적인 기사**:
  - 최근 두 건의 로봇택시 사고에 대한 우려가 제기되며 신뢰성 문제로 이어짐.
  - 내부자들이 주식을 매각하여 시장의 신뢰도에 다소 부정적인 영향을 미치고 있음.

- **뉴스 감정 점수**: 평균적으로 0.1317로 중립점수에 가까운 설정을 보임.

#### 결론
테슬라는 강력한 재무 성과와 다수의 긍정적인 비즈니스 지표를 보이고 있지만, 최근 뉴스에서 나타난 부정적인 사건들은 비상장 주식

## 회사 개요 툴, 추가 정보 툴

In [14]:
class CompanyOverviewToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyOverviewTool(BaseTool):
    name: Type[str] = "company_overview_Tool"
    description: Type[str] = """
        Use this to get the overview of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyOverviewToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas


class CompanyStockPerformanceToolArgsSchema(BaseModel):
    symbol: str = Field(description="Stock symbol of the company. Exmaple: AAPL, TSLA")

class CompanyStockPerformanceTool(BaseTool):
    name: Type[str] = "company_stock_performance_tool"
    description: Type[str] = """
        Use this to get the weekly performance of a company.
        You should enter a stock symbol
    """

    args_schema: Type[BaseModel] = CompanyStockPerformanceToolArgsSchema
    
    def _run(self, symbol: str) -> Dict[str, any]:
        url = f"https://www.alphavantage.co/query?function=TIME_SERIES_WEEKLY&symbol={symbol}&apikey={alpha_vantage_api_key}"
        response = requests.get(url)
        datas = response.json()
        return datas

In [15]:
tools = [
    StockMarkSymbolSearchTool(), # 회사 심볼 툴
    NewsSentimentTool(), # 회사 최근 뉴스, 민감도 툴
    CompanyIncomeStatementTool(), #회사의 재무재표 손익 툴
    CompanyOverviewTool(), # 회사 개요 툴
    CompanyStockPerformanceTool() # 한 주간의 주가 정보를 알아오는 툴
]

agent = create_agent (
    model="gpt-4o-mini",
    tools=tools,
)

NameError: name 'StockMarkSymbolSearchTool' is not defined

In [44]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """
        You are a veteran Wall Street stock investment expert and a cold-blooded Chief Financial Analyst. 
        Your task is to comprehensively analyze the company's financial overview, income statement, and recent stock price trends to provide sharp, data-driven investment insights.

        [CORE DIRECTIVES]
        1. Avoid ambiguity. Never provide vague or irresponsible answers like "it depends on the investor's choice" or "it is difficult to predict."
        2. Make a definitive call. Based on the retrieved data, you MUST provide a clear and explicit final investment conclusion: choosing exactly one from [BUY / HOLD / SELL].
        3. Maintain a highly professional, objective, and authoritative tone. Back up your conclusion logically using concrete numbers and financial metrics (profitability, growth, and price momentum).
        
        Your analysis will guide critical financial decisions. Be ruthless, objective, and strictly rely on the data provided.
    """),
    ("human", """
        You must use tools to answer this question.
    
        1. Find the stock symbol for {company}.
        2. Retrieve the company's financial overview.
        3. Retrieve the company's income statement.
        4. Retrieve the stock price data (recent price, trend, or performance).
        5. Based on ALL of the following:
        - Financial data
        - Income statement
        - Stock price performance
    
        Analyze whether {company} is a good investment.
    
        Final answer must include:
        - Stock symbol
        - Key financial metrics
        - Income insights (revenue, net income)
        - Stock price trend
        - Investment conclusion
    """),
])

chain = prompt | agent | RunnableLambda(parse_output)

In [46]:
result = chain.invoke({
    "company": "테슬라"
})

GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT

In [ ]:
print(result)